In [ ]:
from skan import draw
from skan import Skeleton, summarize
from skan.csr import make_degree_image
import scipy as sc
import skimage as sk
from skimage.io import imsave, imread
from skimage.util import img_as_ubyte
import numpy as np
import pandas as pd
import os
import re
from glob import glob
import joblib
from sklearn.preprocessing import Normalizer
from sklearn.ensemble import RandomForestClassifier
import napari
import itertools
from bioio import BioImage
from pathlib import Path
from random import randint
from matplotlib import pyplot as plt
from scipy import ndimage as ndi
from skimage.morphology import skeletonize, label
from skimage.measure import regionprops
from tqdm import tqdm
from polyline_features import measure_skeleton_segments
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter, FormatStrFormatter, FuncFormatter
import scipy as sc
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
def read_img(file):
    return np.squeeze(BioImage(file).get_image_data("ZYX", C=1))

def get_pixel_spacing(file):
    ''' 
    Use one image from dataset read in with BioImage without .get_image_data to access pixel spacing
    '''
    img = BioImage(file)
    pixel_spacing = (img.physical_pixel_sizes[1],img.physical_pixel_sizes[2])
    #voxel = (img.physical_pixel_sizes[0],img.physical_pixel_sizes[1],img.physical_pixel_sizes[2])
    return pixel_spacing[0]

#clf = joblib.load('/DISKS/2TB/gkristin/Zebrafish_Desmosomes/models/zebrafish_cell_membranes_V3.joblib')
#sigma_min = 1
#sigma_max = 8

In [ ]:
def get_prediction(file):
    img = read_img(file)
    unstack = np.unstack(img)
    slice_preds = []
    for s in unstack:
        features = sk.feature.multiscale_basic_features(s,sigma_min=sigma_min,sigma_max=sigma_max,channel_axis=None)
        f_flat = features.reshape(-1, features.shape[-1])
        transformer = Normalizer().fit(f_flat)
        norm_f = transformer.transform(f_flat)
        pred = clf.predict(norm_f)
        reshaped_pred = pred.reshape(features.shape[:2])
        slice_preds.append(reshaped_pred)
    pred_stack = np.stack(slice_preds) 
    return img, pred_stack                                   

In [ ]:
def get_prediction_list_comp(file):
    img = read_img(file)
    unstack = np.unstack(img)
    features = [sk.feature.multiscale_basic_features(z,sigma_min=sigma_min,sigma_max=sigma_max,channel_axis=None) for z in unstack]
    f_flat = [f.reshape(-1, f.shape[-1]) for f in features]
    transformers = [Normalizer().fit(ff) for ff in f_flat]
    norm_f = [t.transform(ff) for ff, t in zip(f_flat, transformers)]
    preds = [clf.predict(nf) for nf in norm_f]
    reshaped_preds = [p.reshape(f.shape[:2]) for p,f in zip(preds,features)]
    pred_stack = np.stack(reshaped_preds) 
    return img, pred_stack 

In [ ]:
def fill_holes(label):
    binary = label - 1
    unstack = np.unstack(binary)
    footprint = sk.morphology.footprint_rectangle((6,6))
    remove_small_objs_pre = [sk.morphology.remove_small_objects(u,max_size=700) for u in unstack]
    dilate = [sk.morphology.dilation(p,footprint,mode="constant",cval=1) for p in remove_small_objs_pre]
    holes = [sk.morphology.remove_small_holes(d) for d in dilate]
    erode = [sk.morphology.erosion(h,footprint,mode="constant",cval=0) for h in holes]
    remove_small_objs = [sk.morphology.remove_small_objects(e,max_size=700) for e in erode]
    gauss = [sk.filters.gaussian(o, sigma=2.0, preserve_range=True) for o in remove_small_objs]
    threshold = [g > 0.25 for g in gauss]
    skeletons = [skeletonize(t) for t in threshold]
    return np.stack(threshold), np.stack(skeletons)

def save_things(path,name,mask,skeletons):
    mask_path = os.path.join(path,'masks')
    os.makedirs(mask_path,exist_ok=True)
    skeleton_path = os.path.join(path,'skeletons')
    os.makedirs(skeleton_path,exist_ok=True)
    mask_bit = np.array(mask,dtype='uint8')
    skeletons_bit = np.array(skeletons,dtype='uint8')
    imsave(os.path.join(mask_path,f'{name[:-4]}_mask.tif'),mask_bit,check_contrast=False)
    imsave(os.path.join(skeleton_path,f'{name[:-4]}_skeleton.tif'),skeletons_bit,check_contrast=False)

In [ ]:
files = sorted(glob('/DISKS/2TB/gkristin/Zebrafish_Desmosomes/Images/*/*.czi'))
#imgs = list(map(read_img,files))
pixel_spacing = list(map(get_pixel_spacing,files))

In [ ]:
for file, spacing in zip(files,pixel_spacing):
    print(f'{os.path.basename(file)} pixel spacing: {spacing}')

In [ ]:
round(0.386/pixel_spacing[0])

In [ ]:
scaled_sig = [round(0.386/ps,2) for ps in pixel_spacing]
np.unique(scaled_sig)

In [ ]:
for f in tqdm(files):
    name = os.path.basename(f)
    path = Path(f).parent
    img, preds = get_prediction_list_comp(f)
    mask, skeletons = fill_holes(preds)
    save_things(path,name,mask,skeletons)

In [ ]:
skeleton_files = sorted(glob('/DISKS/2TB/gkristin/Zebrafish_Desmosomes/Images/*/skeletons/*.tif'))

degree_imgs = []
for sf in skeleton_files:
    path = Path(sf).parents[1]
    name = os.path.basename(sf)[:-4]
    new_name = name.replace('skeleton','deg_img')
    os.makedirs(os.path.join(path,'degree_imgs'),exist_ok=True)
    skimg = imread(sf)
    sk_unstack = np.unstack(skimg)
    deg_imgs = list(map(make_degree_image,sk_unstack))
    degree_imgs.append(deg_imgs)
    deg_stack = np.stack(deg_imgs,dtype='int8')
    imsave(os.path.join(path,'degree_imgs',f'{new_name}.tif'),deg_stack,check_contrast=False)


In [ ]:
# Getting measurements for skeleton components, testing function from Claude
skeleton_files = sorted(glob('/DISKS/2TB/gkristin/Zebrafish_Desmosomes/Images/*/degree_imgs/*.tif'))
org_files = sorted(glob('/DISKS/2TB/gkristin/Zebrafish_Desmosomes/Images/*/*.czi'))
basenames = list(map(os.path.basename,skeleton_files))
names = [name[0:-4].replace('_Out_deg_img', '') for name in basenames]
batches = [re.search(r"Batch_\d{1}",file).group() for file in skeleton_files]
pixel_spacing = list(map(get_pixel_spacing,org_files))
c = 0
all_dfs = []
for name, batch, file, spacing in zip(names,batches,skeleton_files,pixel_spacing):
    path = os.path.join(Path(file).parents[1],'filtered_segments')
    os.makedirs(path,exist_ok=True)
    condition = 'WT_ctrl' if 'WT' in name else 'DKO_ctrl'
    fish_id = f'{name.split('_')[2]}_{condition}_{batch}'
    deg_img = imread(file)
    filtered_deg_img = deg_img == 2
    unstacked = np.unstack(filtered_deg_img)
    dfs = []
    scaled_sigma = round(0.386/spacing,2)
    for i, s in enumerate(unstacked):
        new_name = f'{name}_z{i}'
        df = measure_skeleton_segments(s, min_pixels=30,path=path,name=new_name,pixel_size=spacing,sigma=scaled_sigma)
        df["z-slice"] = i
        df["image"] = name
        df["batch"] = batch
        df["condition"] = condition
        df["fish_id"] = fish_id
        dfs.append(df)
    merged_dfs = pd.concat(dfs,ignore_index=True)
    all_dfs.append(merged_dfs)
all_data = pd.concat(all_dfs,ignore_index=True)
all_data.to_csv('/DISKS/2TB/gkristin/Zebrafish_Desmosomes/Images/Polyline_Analysis_All_Data.csv',index=False)

In [ ]:
v = napari.Viewer()

In [ ]:
deg_stack = np.stack(degree_imgs[0])
v.add_labels(deg_stack == 2)

In [ ]:
all_data = pd.read_csv('/DISKS/2TB/gkristin/Zebrafish_Desmosomes/Images/Polyline_Analysis_All_Data.csv')
median_img = all_data.groupby(['image','fish_id','batch','condition'],as_index=False).median(numeric_only=True)
median_fish = median_img.groupby(['fish_id','batch','condition'],as_index=False).median(numeric_only=True)

In [ ]:
median_fish.columns

In [ ]:
import seaborn.objects as so

In [ ]:
sns.pointplot(data=median_fish,x='condition',y='max_chord_deviation',order=['WT_ctrl','DKO_ctrl'],estimator="mean",errorbar=("ci",95),linestyles="none",color='black',capsize=0.15,scale=0.2,zorder=3)
sns.swarmplot(data=median_fish,x='condition',y='max_chord_deviation',order=['WT_ctrl','DKO_ctrl'],hue='condition',palette=['#e7fbc9','#ebe673'],edgecolor='gray',linewidth=0.2,size=10,zorder=1)


In [ ]:
#bracket params
x1, x2 = 0,1
y_base = 1.021
bracket_tick = .001
plt.figure()
ax = sns.swarmplot(data=median_fish,x='condition',y='tortuosity',order=['WT_ctrl','DKO_ctrl'],hue='condition',palette=['#e7fbc9','#ebe673'],edgecolor='black',linewidth=0.2,size=5,zorder=1)
sns.pointplot(data=median_fish,x='condition',y='tortuosity',order=['WT_ctrl','DKO_ctrl'],estimator="mean",errorbar=("ci",95),linestyles="none",color='black',capsize=0.15,scale=0.3,zorder=3)
#axs[0].set_title('Tortuosity')
ax.set_ylabel('Tortuosity (length/chord)')
ax.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))
ax.xaxis.set_ticklabels(['WT','DKO'])
ax.xaxis.set_label_text('')
ax.set_xbound()
ax.spines.right.set_visible(False)
ax.spines.top.set_visible(False)
ax.set_xlim(-0.5, 1.5) # Hard limits stop any spacing changes
ax.set_ylim(top=y_base + 0.05)
#plt.tight_layout()
plt.show()

In [ ]:
def format_func(value, tick_number):
    return f'{value:.4f}'


fig, axs = plt.subplots(1,3,figsize=(7,3))
sns.swarmplot(data=median_fish,x='condition',y='tortuosity',order=['WT_ctrl','DKO_ctrl'],hue='condition',palette=['#e7fbc9','#ebe673'],edgecolor='black',linewidth=0.2,size=6,zorder=1,ax=axs[0])
sns.pointplot(data=median_fish,x='condition',y='tortuosity',order=['WT_ctrl','DKO_ctrl'],estimator="mean",errorbar=("ci",95),linestyles="none",color='black',capsize=0.15,scale=0.3,zorder=3,ax=axs[0])
#axs[0].set_title('Tortuosity')
axs[0].set_ylabel('Tortuosity (length/chord)')
axs[0].yaxis.set_major_formatter(FormatStrFormatter('%.3f'))
axs[0].xaxis.set_ticklabels(['WT','DKO'])
axs[0].xaxis.set_label_text('')
axs[0].spines.right.set_visible(False)
axs[0].spines.top.set_visible(False)
sns.swarmplot(data=median_fish,x='condition',y='max_chord_deviation',order=['WT_ctrl','DKO_ctrl'],hue='condition',palette=['#e7fbc9','#ebe673'],edgecolor='black',linewidth=0.2,size=6,zorder=1,ax=axs[1])
sns.pointplot(data=median_fish,x='condition',y='max_chord_deviation',order=['WT_ctrl','DKO_ctrl'],estimator="mean",errorbar=("ci",95),linestyles="none",color='black',capsize=0.15,scale=0.3,zorder=3,ax=axs[1])
#axs[1].set_title('Max Chord Deviation')
axs[1].set_ylabel('Distance from chord (um)')
axs[1].yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
axs[1].xaxis.set_ticklabels(['WT','DKO'])
axs[1].xaxis.set_label_text('')
axs[1].spines.right.set_visible(False)
axs[1].spines.top.set_visible(False)
sns.swarmplot(data=median_fish,x='condition',y='total_abs_turning',order=['WT_ctrl','DKO_ctrl'],hue='condition',palette=['#e7fbc9','#ebe673'],edgecolor='black',linewidth=0.2,size=6,zorder=1,ax=axs[2])
sns.pointplot(data=median_fish,x='condition',y='total_abs_turning',order=['WT_ctrl','DKO_ctrl'],estimator="mean",errorbar=("ci",95),linestyles="none",color='black',capsize=0.15,scale=0.3,zorder=3,ax=axs[2])
#axs[2].set_title('Total Turning')
axs[2].set_ylabel('Abs. Total Turning (radians)')
axs[2].xaxis.set_ticklabels(['WT','DKO'])
axs[2].xaxis.set_label_text('')
axs[2].spines.right.set_visible(False)
axs[2].spines.top.set_visible(False)
#axs[2].xaxis.set_major_formatter(StrMethodFormatter('{x:.2f}'))
#plt.gca().yaxis.set_major_formatter(FuncFormatter(format_func))
plt.tight_layout()
plt.savefig('/DISKS/2TB/gkristin/Zebrafish_Desmosomes/Figure_Shape_Features_Fish_Medians_V5.svg',dpi=300)
plt.show()

In [ ]:
# create plots for easy vis
fig, axs = plt.subplots(1,7,figsize=(20,5))
sns.violinplot(data=median_fish, x='condition', y='tortuosity', hue='batch', split='batch',gap=0.15,fill=False, hue_order=['Batch_1','Batch_2'],inner='stick',ax=axs[0])
axs[0].set_title('tortuosity')
axs[0].set_ylabel('tortuosity')
sns.violinplot(data=median_fish, x='condition', y='straightness',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['Batch_1','Batch_2'],inner='stick',ax=axs[1])
axs[1].set_title('straightness')
axs[1].set_ylabel('straightness')
sns.violinplot(data=median_fish, x='condition', y='max_chord_deviation',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['Batch_1','Batch_2'],inner='stick',ax=axs[2])
axs[2].set_title('max_chord_deviation')
axs[2].set_ylabel('max_chord_deviation')
sns.violinplot(data=median_fish, x='condition', y='median_abs_curvature',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['Batch_1','Batch_2'],inner='stick',ax=axs[3])
axs[3].set_title('median_abs_curvature')
axs[3].set_ylabel('median_abs_curvature')
sns.violinplot(data=median_fish, x='condition', y='total_abs_turning',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['Batch_1','Batch_2'],inner='stick',ax=axs[4])
axs[4].set_title('total_abs_turning')
axs[4].set_ylabel('total_abs_turning')
sns.violinplot(data=median_fish, x='condition', y='soam',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['Batch_1','Batch_2'],inner='stick',ax=axs[5])
axs[5].set_title('soam')
axs[5].set_ylabel('total turns / length')
sns.violinplot(data=median_fish, x='condition', y='n_inflections',hue='batch',split='batch',gap=0.15,fill=False, hue_order=['Batch_1','Batch_2'],inner='stick',ax=axs[6])
axs[6].set_title('n_inflections')
axs[6].set_ylabel('n_inflections')
plt.tight_layout()
#plt.savefig('/DISKS/2TB/gkristin/Zebrafish_Desmosomes/Polyline_Shape_Features_Fish_Medians.png',dpi=300)
plt.show()

In [ ]:
batch1_DKO = median_fish.query('batch == "Batch_1" & condition == "DKO_ctrl"')
batch2_DKO = median_fish.query('batch == "Batch_2" & condition == "DKO_ctrl"')
batch1_WT = median_fish.query('batch == "Batch_1" & condition == "WT_ctrl"')
batch2_WT = median_fish.query('batch == "Batch_2" & condition == "WT_ctrl"')

wt = median_fish.query('condition == "WT_ctrl"')
dko = median_fish.query('condition == "DKO_ctrl"')

In [ ]:
median_fish.columns

In [ ]:
#testing batch differences:
wt_tort_batches = sc.stats.ttest_ind(batch1_WT['tortuosity'].to_numpy(),batch2_WT['tortuosity'].to_numpy(),equal_var=False)
wt_dev_batches = sc.stats.ttest_ind(batch1_WT['max_chord_deviation'].to_numpy(),batch2_WT['max_chord_deviation'].to_numpy(),equal_var=False)
wt_turn_batches = sc.stats.ttest_ind(batch1_WT['total_abs_turning'].to_numpy(),batch2_WT['total_abs_turning'].to_numpy(),equal_var=False)
dko_tort_batches = sc.stats.ttest_ind(batch1_DKO['tortuosity'].to_numpy(),batch2_DKO['tortuosity'].to_numpy(),equal_var=False)
dko_dev_batches = sc.stats.ttest_ind(batch1_DKO['max_chord_deviation'].to_numpy(),batch2_DKO['max_chord_deviation'].to_numpy(),equal_var=False)
dko_turn_batches = sc.stats.ttest_ind(batch1_DKO['total_abs_turning'].to_numpy(),batch2_DKO['total_abs_turning'].to_numpy(),equal_var=False)

In [ ]:
print(f"""
    p-values from the following tests:
    WT batches Tort Welch's T-test: {wt_tort_batches.pvalue}
    WT batches Deviation Welch's T-test: {wt_dev_batches.pvalue}
    WT batches Turns Welch's T-test: {wt_turn_batches.pvalue}
    DKO batches Tort Welch's T-test: {dko_tort_batches.pvalue}
    DKO batches Deviation Welch's T-test: {dko_dev_batches.pvalue}
    DKO batches Turns Welch's T-test: {dko_turn_batches.pvalue}
"""
    )

In [ ]:
wt_v_dko_tort = sc.stats.ttest_ind(wt['tortuosity'].to_numpy(),dko['tortuosity'].to_numpy(),equal_var=False)
wt_v_dko_dev = sc.stats.ttest_ind(wt['max_chord_deviation'].to_numpy(),dko['max_chord_deviation'].to_numpy(),equal_var=False)
wt_v_dko_turn = sc.stats.ttest_ind(wt['total_abs_turning'].to_numpy(),dko['total_abs_turning'].to_numpy(),equal_var=False)
print(f"""
    p-values from the following tests:
    WT v DKO Tort Welch's T-test: {wt_v_dko_tort.pvalue}
    WT v DKO Dev Welch's T-test: {wt_v_dko_dev.pvalue}
    WT v DKO Turn Welch's T-test: {wt_v_dko_turn.pvalue}
"""
    )

In [ ]:
tort_mod = smf.ols(formula='tortuosity ~ condition + batch',data=median_fish)
dev_mod = smf.ols(formula='max_chord_deviation ~ condition + batch',data=median_fish)
turn_mod = smf.ols(formula='total_abs_turning ~ condition + batch',data=median_fish)

In [ ]:
tort_res = tort_mod.fit()
dev_res = dev_mod.fit()
turn_res = turn_mod.fit()

In [ ]:
print(f"""
Tort
{tort_res.pvalues} \n
Deviation
{dev_res.pvalues} \n
Abs Turning
{turn_res.pvalues} \n
""")

In [ ]:
print(f"""
Tort
{tort_res.conf_int(alpha=.05)} \n
Deviation
{dev_res.conf_int(alpha=.05)} \n
Abs Turning
{turn_res.conf_int(alpha=.05)} \n
""")

In [ ]:
print(f"""
Max Deviation
{dev_res.summary2()} \n
======================================================= \n
Confidence Interval \n
{dev_res.conf_int(alpha=.05)}
======================================================= \n
P-Value
{dev_res.pvalues}
""")

In [ ]:
# Source - https://stackoverflow.com/a/51734632
# Posted by Michael B
# Retrieved 2026-07-21, License - CC BY-SA 4.0

def results_summary_to_dataframe(results):
    '''take the result of an statsmodel results table and transforms it into a dataframe'''
    pvals = results.pvalues
    coeff = results.params
    conf_lower = results.conf_int()[0]
    conf_higher = results.conf_int()[1]

    results_df = pd.DataFrame({"pvals":pvals,
                               "coeff":coeff,
                               "conf_lower":conf_lower,
                               "conf_higher":conf_higher
                                })

    #Reordering...
    results_df = results_df[["coeff","pvals","conf_lower","conf_higher"]]
    return results_df

In [ ]:
print(f"""
Tortuosity
{tort_res.summary2()} \n
======================================================= \n
Confidence Interval \n
{tort_res.conf_int(alpha=.05)}
======================================================= \n
======================================================= \n
Straightness
{strt_res.summary2()} \n
======================================================= \n
Confidence Interval \n
{strt_res.conf_int(alpha=.05)}
======================================================= \n
======================================================= \n
Total Absolute Turning
{turn_res.summary2()} \n
======================================================= \n
Confidence Interval \n
{turn_res.conf_int(alpha=.05)}
""")

## Testing Cells

In [ ]:
imgs = []
preds = []
i = randint(0,len(files))
#img, pred = get_prediction(files[i])

#for f in files:
#    img, pred = get_prediction(f)
#    imgs.append(img)
#    preds.append(pred)
#    break

In [ ]:
img, pred = get_prediction_list_comp(files[i])

In [ ]:
v = napari.Viewer()

In [ ]:
v.add_image(imgs[0])
v.add_labels(preds[0]==2)

In [ ]:
pred = preds[0]
img = imgs[0]

In [ ]:
binary = pred - 1
binary_unstack = np.unstack(binary)

In [ ]:
v.add_labels(binary)

In [ ]:
footprint = sk.morphology.footprint_rectangle((6,6))
remove_small_objs_pre = [sk.morphology.remove_small_objects(z,max_size=700) for z in binary_unstack]
dilate = [sk.morphology.dilation(p,footprint,mode="constant",cval=1) for p in remove_small_objs_pre]
holes = [sk.morphology.remove_small_holes(d) for d in dilate]
erode = [sk.morphology.erosion(h,footprint,mode="constant",cval=0) for h in holes]
remove_small_objs = [sk.morphology.remove_small_objects(e,max_size=700) for e in erode]


In [ ]:
dilate_stack = np.stack(dilate)
holes_stack = np.stack(holes)
erode_stack = np.stack(erode)
objs_stack = np.stack(remove_small_objs)

In [ ]:
#v.add_labels(dilate_stack)
#v.add_labels(holes_stack)
#v.add_labels(erode_stack)
v.add_image(img)
v.add_labels(objs_stack)

In [ ]:
skeleton = [skeletonize(z) for z in remove_small_objs]
skeleton_stack = np.stack(skeleton)

In [ ]:
v.add_labels(skeleton_stack)

In [ ]:
v.add_image(img)

In [ ]:
add = [z * 255 for z in remove_small_objs]

In [ ]:
gauss = [sk.filters.gaussian(o, sigma=2.0, preserve_range=True) for o in remove_small_objs]
gauss_stack = np.stack(gauss)

In [ ]:
v.add_image(gauss_stack)

In [ ]:
threshold = gauss_stack > 0.25

In [ ]:
unstack_threshold = np.unstack(threshold)

In [ ]:
skeleton_gauss = [skeletonize(t) for t in unstack_threshold]
skeleton_gauss_stack = np.stack(skeleton_gauss)

In [ ]:
#v.add_labels(threshold)
v.add_labels(skeleton_gauss_stack)
#v.add_labels(skeleton_stack)

In [ ]:
len(skeleton_gauss)

In [ ]:
scale = pixel_spacing[0][0]
scale

In [ ]:
skeleton_summary = [summarize(Skeleton(s,spacing=scale),separator='_') for s in skeleton_gauss]

In [ ]:
skeleton_summary[0].head()

In [ ]:
degree_images = list(map(make_degree_image,skeleton_gauss))

In [ ]:
degree_images_stack = np.stack(degree_images)

In [ ]:
v.add_labels(degree_images_stack)

In [ ]:
v.add_image(img)